# Tutorial 1: Quickstart Scientific Taste

Estimated time: 20-30 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: run `validate -> plan -> run -> runs list/show` on a typed spec
- Secondary scientific aim: relate input perturbations (`a`, `b`) to output behavior (`y`)

## Success criteria
- you can explain one observed trend in outputs and point to its run artifacts


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Validate and plan

In [ ]:
run_cli('validate', 'tutorials/specs/model.toy.grid.json')
run_cli('plan', 'tutorials/specs/model.toy.grid.json')


## Step 2: Execute runs and list run IDs

In [ ]:
run_cli('run', 'tutorials/specs/model.toy.grid.json')
run_cli('runs', 'list')


## Step 3: Auto-select a recent run and inspect it

In [ ]:
import json

reg_path = ROOT / 'tmp/run_registry.json'
registry = json.loads(reg_path.read_text())

candidates = []
for run_id, record in registry.items():
    if isinstance(record, str):
        run_payload = json.loads((ROOT / record).read_text())
    else:
        run_payload = record
    inputs_path = run_payload.get('inputs_path', '')
    if inputs_path.startswith('tmp/tutorials/toy_store/runs/'):
        candidates.append((run_payload.get('finished_at', ''), run_id))

if not candidates:
    raise RuntimeError('No tutorial toy runs found. Run Step 2 first.')

run_id = sorted(candidates)[-1][1]
print('Using run id:', run_id)
run_cli('runs', 'show', run_id)


## Step 4: Visualize response surface (graphic)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

runs_root = ROOT / 'tmp/tutorials/toy_store/runs'
rows = []
for run_dir in sorted(p for p in runs_root.iterdir() if p.is_dir()):
    inp = json.loads((run_dir / 'inputs.json').read_text())
    out = json.loads((run_dir / 'outputs.json').read_text())
    y0 = out['y']['y'][0]
    rows.append((inp['a'], inp['b'], y0))

arr = np.array(rows, dtype=float)
plt.figure(figsize=(5, 4))
sc = plt.scatter(arr[:, 0], arr[:, 1], c=arr[:, 2], s=120, cmap='viridis')
plt.xlabel('a')
plt.ylabel('b')
plt.title('Toy model first output component y[0]')
plt.colorbar(sc, label='y[0]')
plt.grid(True, alpha=0.3)
plt.show()


## Scientific checkpoint
- Describe one monotonic trend you see in the plot.
- Explain whether the trend matches your expectation from the toy equation.


## Common mistakes
- Running before installing the package — `mm doctor` will flag this.
- Running cells out of order — bootstrap first, then steps in sequence.
